# 208. Implement Trie (Prefix Tree)
**Difficulty:** 🟡 Medium · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/implement-trie-prefix-tree/

## 💡 Concepts

**Core concept(s):** Build a **trie** — a tree of characters where shared prefixes share a path.

**Why it applies here:** We need fast `insert`, whole-word `search`, and "any word start with this prefix?" (`startsWith`). A trie answers all three in time proportional to the word length, no matter how many words are stored — something a plain set can't do for prefixes.

**Key intuition:** Walk the word letter by letter down the tree, creating nodes as needed; a marker says "a word ends here".

---

### 📚 What is a Trie (Prefix Tree)?
A **trie** stores words letter-by-letter along paths from a root, so words sharing a prefix share the same early path. A marker flags where a word ends.
- **Complexity:** insert / search a word of length L is **O(L)**, no matter how many words are stored.
- **In Python:** nested `dict`s (`{char: child}`) with a sentinel like `'$'` for word-ends.

---

**Prerequisite knowledge:**
- Nested dictionaries.
- Why a set is fine for exact search but poor for prefix search.

## 📝 Problem

Implement `insert(word)`, `search(word)` (exact), and `startsWith(prefix)`.

**Example**
```
insert("apple"); search("apple") -> True; search("app") -> False; startsWith("app") -> True
```

> Two approaches: a naive set (slow prefix) and a trie (fast prefix).

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    """Level-order list -> tree (None = missing child), LeetCode style."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values):
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (in-order sorted, height ~log n)."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)
        node.right = helper(mid + 1, hi)
        return node
    return helper(1, n)

def preorder(root):
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    if not a and not b: return True
    if not a or not b or a.val != b.val: return False
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Set of Words (naive prefix)

**Idea:** Store words in a set. `search` is instant, but `startsWith` must scan every word.

**Time:** insert/search `O(L)`; **startsWith `O(N·L)`** (scans all words).

**Space:** `O(total letters)`.

In [ ]:
class TrieSetNaive:
    """A naive dictionary: exact search is fast, but prefix search scans every word."""
    def __init__(self):
        self.words = set()                 # just store all inserted words
    def insert(self, word: str) -> None:
        self.words.add(word)
    def search(self, word: str) -> bool:
        return word in self.words          # O(1) exact lookup
    def startsWith(self, prefix: str) -> bool:
        return any(w.startswith(prefix) for w in self.words)  # must scan EVERY word (slow)

### Approach 2 — Trie (optimal)

**Idea:** Store letters along paths. Each operation just walks the prefix; `startsWith` is `O(L)` regardless of how many words exist.

**Time:** insert/search/startsWith all `O(L)`.

**Space:** `O(total letters)`.

In [ ]:
class Trie:
    """A prefix tree: words share their common starting path, letter by letter."""
    def __init__(self):
        self.root = {}                     # each node is a dict: {letter: child_node}
    def insert(self, word: str) -> None:
        node = self.root
        for c in word:
            node = node.setdefault(c, {})  # follow the letter path, creating nodes as needed
        node["$"] = True                   # mark that a complete word ends here
    def search(self, word: str) -> bool:
        node = self._walk(word)            # walk the exact letters
        return node is not None and "$" in node   # word exists only if it's marked as an end
    def startsWith(self, prefix: str) -> bool:
        return self._walk(prefix) is not None      # some word has this prefix if the path exists
    def _walk(self, s):                    # follow the path for string s; None if it breaks
        node = self.root
        for c in s:
            if c not in node:
                return None                # no such path -> not present
            node = node[c]
        return node

In [ ]:
# Correctness check
for T in (TrieSetNaive, Trie):
    t = T()
    t.insert("apple")
    assert t.search("apple") is True
    assert t.search("app") is False
    assert t.startsWith("app") is True
    t.insert("app")
    assert t.search("app") is True
    assert t.startsWith("apx") is False
    print(T.__name__, "OK")
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on a growing number of words `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **same word set for both, then run one `startsWith` per word.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def _words(n):
    # distinct lowercase words, length ~5
    out = []
    for i in range(n):
        s = ""
        x = i
        for _ in range(5):
            s += chr(ord("a") + x % 26); x //= 26
        out.append(s)
    return out

def naive_bulk(words):
    t = TrieSetNaive()
    for w in words: t.insert(w)
    return sum(t.startsWith(w[:3]) for w in words)   # N prefix queries

def trie_bulk(words):
    t = Trie()
    for w in words: t.insert(w)
    return sum(t.startsWith(w[:3]) for w in words)

def make_worst_case(n):
    return (_words(n),)

solutions = {
    "set  (prefix O(N^2))": naive_bulk,
    "trie (prefix O(N)  )": trie_bulk,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Trie for prefix work:** any "words sharing a start" task (autocomplete, prefix counts) becomes `O(L)` per query.
- **A set is not enough:** exact lookup yes, but prefix questions force a full scan.
- **Signal:** "prefix", "startsWith", "autocomplete", "dictionary of words".
- **Related problems:** Add and Search Word, Word Search II, Replace Words.
- **Common pitfalls:** (1) forgetting the end-of-word marker (then "app" matches inside "apple"); (2) using a set and getting slow prefix queries.